# SignBridge AI - Exploratory data analysis
Works on any landmark file in `data/processed/`. Real files are preferred; if none exist the **synthetic demo** files are used and every plot is labelled accordingly.

In [ ]:
import sys, json, glob
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from core.dataset import LandmarkSet
from core.normalize import normalize_hands, static_features, sequence_features

LANG, KIND = "ASL", "static"
real = [f for f in glob.glob(str(ROOT / "data/processed" / f"*_{LANG}_{KIND}.npz")) if not Path(f).name.startswith("demo_")]
files = real or glob.glob(str(ROOT / "data/processed" / f"demo_{LANG}_{KIND}.npz"))
SYNTH = not real
ds = LandmarkSet.load(files[0])
print(files[0], "| samples:", len(ds), "| SYNTHETIC" if SYNTH else "")

## Class balance and signers

In [ ]:
counts = pd.Series(ds.y).value_counts().sort_index()
ax = counts.plot.bar(figsize=(8, 3), title="Samples per class" + (" (SYNTHETIC)" if SYNTH else ""))
plt.tight_layout(); plt.show()
print("imbalance ratio (max/min):", round(counts.max() / counts.min(), 2))
print("distinct signers:", len(set(ds.signer)), "-> signer-independent split possible" if len(set(ds.signer)) >= 3 else "-> NOT possible (random split fallback)")
print(pd.crosstab(ds.signer, ds.y).head(10))

## Hands present per sample and raw vs normalised coordinates

In [ ]:
print("hands per sample:", pd.Series(ds.mask.sum(-1)).value_counts().to_dict())
i = np.where(ds.mask[:, 0])[0][:200]
raw_wrist_spread = ds.raw[i, 0, 0, :2].std(0)
norm = normalize_hands(ds.raw[i], ds.mask[i])[:, 0]
print("wrist position std BEFORE normalisation (x,y):", raw_wrist_spread.round(3))
print("wrist position std AFTER  normalisation (x,y):", norm[:, 0, :2].std(0).round(3), "(always 0: wrist is the origin)")
print("palm length after normalisation (should be 1):", np.linalg.norm(norm[:, 9], axis=1).mean().round(3))

## Mean normalised hand per class

In [ ]:
from core.landmarks import HAND_CONNECTIONS
classes = sorted(set(ds.y))
cols = 4; rows = int(np.ceil(len(classes) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
nh = normalize_hands(ds.raw, ds.mask)
for ax, c in zip(np.ravel(axes), classes):
    sel = (ds.y == c) & (ds.mask[:, 0] | ds.mask[:, 1])
    h = nh[sel][:, 0].mean(0) if sel.any() else np.zeros((21, 3))
    for a, b in HAND_CONNECTIONS:
        ax.plot([h[a, 0], h[b, 0]], [-h[a, 1], -h[b, 1]], "-", color="tab:green")
    ax.scatter(h[:, 0], -h[:, 1], s=8, color="k"); ax.set_title(c); ax.set_aspect("equal"); ax.axis("off")
for ax in np.ravel(axes)[len(classes):]: ax.axis("off")
fig.suptitle("Mean normalised hand per class" + (" (SYNTHETIC)" if SYNTH else "")); plt.tight_layout(); plt.show()

## Do the classes separate? (PCA of static features)

In [ ]:
from sklearn.decomposition import PCA
X = static_features(ds.raw, ds.mask)
Z = PCA(2, random_state=0).fit_transform(X)
plt.figure(figsize=(6, 5))
for c in classes:
    m = ds.y == c; plt.scatter(Z[m, 0], Z[m, 1], s=6, label=c)
plt.legend(markerscale=2, fontsize=7); plt.title("PCA of normalised landmark features" + (" (SYNTHETIC)" if SYNTH else "")); plt.show()